# Notebook 01: Preparación y Validación del Dataset de Entrenamiento


---
### Objetivo
Construir el dataset de entrenamiento con al menos 2,000 pares pregunta-SQL en español, asegurando que el 100% de las consultas se ejecuten sin error en MySQL 8.4.10 bajo el modo estricto (ONLY_FULL_GROUP_BY). Se documentan y analizan los motivos de descarte de las consultas no admitidas.

In [5]:
# ── Celda 0: Setup universal (Local en Mac o Colab T4 desde VS Code) ──
import os, sys, subprocess
from pathlib import Path

# Detectar si estamos en Google Colab
EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if EN_COLAB:
    import getpass
    REPO = "Albu231311/Deep-Proyecto3"
    NOMBRE = "Albu231311"
    CORREO = "alb231311@uvg.edu.gt"
    if not os.path.exists("/content/repo"):
        token = getpass.getpass("GitHub token: ")
        subprocess.run(f"git clone https://{token}@github.com/{REPO}.git /content/repo", shell=True, check=True)
    os.chdir("/content/repo")
    subprocess.run("git pull", shell=True, check=True)
    subprocess.run(f'git config user.name "{NOMBRE}"', shell=True)
    subprocess.run(f'git config user.email "{CORREO}"', shell=True)
    subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)
    ROOT_DIR = Path("/content/repo")
else:
    # Si el kernel inició dentro de notebooks/, cambiar el directorio de trabajo a la raíz
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
    ROOT_DIR = Path.cwd().resolve()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print(f" Entorno listo. Directorio de trabajo: {os.getcwd()} | En Colab: {EN_COLAB}")


 Entorno listo. Directorio de trabajo: /Users/carlitos/Documents/VIII Semestre/Deep Learning/Deep-Proyecto3 | En Colab: False


## 1. Exploración de MultiSpider en Español

Cargamos las preguntas de Spider traducidas al español neutro.

In [6]:
import json
from pathlib import Path

# Resolver ruta robusta (sea que el kernel esté en la raíz o en notebooks/)
train_es_path = Path("datos/spider/train_es.json")
if not train_es_path.exists() and Path("../datos/spider/train_es.json").exists():
    train_es_path = Path("../datos/spider/train_es.json")

with open(train_es_path, "r", encoding="utf-8") as f:
    spider_data = json.load(f)

print(f"Total de ejemplos en MultiSpider: {len(spider_data)}")
ejemplo = spider_data[0]
print("Ejemplo:")
print(f"  Base de datos: {ejemplo['db_id']}")
print(f"  Pregunta (ES): {ejemplo['question']}")
print(f"  Query original (SQLite): {ejemplo['query']}")


Total de ejemplos en MultiSpider: 8659
Ejemplo:
  Base de datos: department_management
  Pregunta (ES): ¿Cuántos jefes de departamento tienen más de 56 años?
  Query original (SQLite): SELECT count(*) FROM head WHERE age  >  56


## 2. Transpilación SQLite a MySQL 8.4 con sqlglot

Se normalizan identificadores y operadores para asegurar compatibilidad con MySQL 8.4.

In [7]:
import sqlglot

sql_sqlite = ejemplo["query"]
sql_mysql = sqlglot.transpile(sql_sqlite, read="sqlite", write="mysql")[0]
print("SQLite :", sql_sqlite)
print("MySQL  :", sql_mysql)


SQLite : SELECT count(*) FROM head WHERE age  >  56
MySQL  : SELECT COUNT(*) FROM head WHERE age > 56


## 3. Validación de Ejecución con Usuario de Solo Lectura

Se prueba la conexión al motor MySQL 8.4 con el usuario restringido evaluador.

In [8]:
from src.mysql_utils import ejecutar

res = ejecutar("SELECT 1 AS conexion_activa", "sistema_academico")
print("Estado de ejecución en MySQL:", res)


Estado de ejecución en MySQL: {'ok': True, 'columnas': ['conexion_activa'], 'filas': [[1]], 'error': None}


## 4. Dataset de Entrenamiento Final y Resumen de Descartes

Se lee train.jsonl (≥2,400 pares) y analizamos la distribución de dificultad.

In [9]:
from collections import Counter

train_path = Path("datos/entrenamiento/train.jsonl")
if not train_path.exists() and Path("../datos/entrenamiento/train.jsonl").exists():
    train_path = Path("../datos/entrenamiento/train.jsonl")

with open(train_path, "r", encoding="utf-8") as f:
    train_pairs = [json.loads(line) for line in f]

niveles = Counter(p["nivel"] for p in train_pairs)
print(f"Total de pares válidos aceptados: {len(train_pairs)}")
for niv, c in niveles.items():
    print(f"  Nivel {niv.capitalize()}: {c} ({c / len(train_pairs) * 100:.1f}%)")


Total de pares válidos aceptados: 2429
  Nivel Facil: 1037 (42.7%)
  Nivel Dificil: 382 (15.7%)
  Nivel Media: 1010 (41.6%)


## 5. Chequeo de Fuga de Datos

Se ejecuta src/chequeo_fuga.py para asegurar que los esquemas de validación no existan en entrenamiento.

In [ ]:
from src.chequeo_fuga import extraer_tablas_y_columnas, verificar_fuga, extraer_info_de_train

val_dir = Path("datos/validacion/esquemas_val")
if not val_dir.exists() and Path("../datos/validacion/esquemas_val").exists():
    val_dir = Path("../datos/validacion/esquemas_val")

esquemas_val = {}
for val_sql in val_dir.glob("*.sql"):
    with open(val_sql) as f:
        esquemas_val[val_sql.stem] = extraer_tablas_y_columnas(f.read())

tablas_train = extraer_info_de_train(str(train_path))
hay_fuga, reporte = verificar_fuga(esquemas_val, tablas_train)
print(reporte)
assert not hay_fuga, " Error: se detectó fuga de datos"


✅ Sin fuga: ningún esquema de validación coincide con el entrenamiento.
